In [1]:
# 1. Importación de librerías esenciales
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [2]:
# 2. Cargar el dataset del Titanic
df = pd.read_csv('titanic.csv')

In [3]:
# 3. Inspeccionar las primeras filas para confirmar que cargó correctamente
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Revisar estructura general
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


# Analisis de la extructura
Como se ve hay datos faltantes que en el caso por ejemplo de Cabina son muchos. Como si que me gustaria trabajar con estos datos la forma mas inteligente seria usarlos para extraer la cubierta y poder analizar aunque no sea de forma muy precisa la distribución de precios y supervivencia según la altura de la cubierta (*Deck* A, B, C, D, E, F, G) añadiendo una 'U' para el valor desconocido faltante.
El caso del embarque es un poco distinto son solo 2 los datos faltantes asi que rellenar con la moda no afectara demasiado.
En la edad tambien faltan bastantes datos usaremos una mediana agrupada por titulo social con el fin de intertar dar un valor de edad media mas adecuado al inidividuo.

In [12]:
# Corrigiendo y tratando los datos

# 1. Imputar 'Embarked' con la moda
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)

# 2. Crear la columna 'Deck' desde 'Cabin' para la Pestaña 3
df['Deck'] = df['Cabin'].apply(lambda x: x[0] if pd.notnull(x) else 'U')

# 3. Extraer 'Title' de la columna 'Name' usando una raw string (r'...')
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Agrupar títulos poco frecuentes
mapa_titulos = {
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'
}
df['Title'] = df['Title'].replace(mapa_titulos)

# 4. Imputar 'Age' con la mediana según el 'Title'
df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# Verificación de que ya no hay nulos
print("Nulos restantes por columna:")
print(df[['Age', 'Embarked', 'Deck']].isnull().sum())

Nulos restantes por columna:
Age         0
Embarked    0
Deck        0
dtype: int64


In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,U,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,C,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,U,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,C,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,U,Mr


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     891 non-null    str    
 12  Deck         891 non-null    str    
 13  Title        891 non-null    str    
dtypes: float64(2), int64(5), str(7)
memory usage: 136.0 KB


# Preparando variables
Ya tenemos los datos limpios para empezar a trabajar. 'Cabin' sigue solo con 204 no nulos pero no la usaremos. Las primeras variables a crear son para Tarifa Real y Estructura Familiar

In [13]:
# 1. Tamaño del grupo del billete y Tarifa Real por Persona
df['GroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')
df['Fare_Per_Person'] = df['Fare'] / df['GroupSize']

# 2. Estructura familiar
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Inspeccionar las nuevas columnas creadas
df[['Ticket', 'Fare', 'GroupSize', 'Fare_Per_Person', 'FamilySize', 'IsAlone', 'Deck']].head(10)

# isAlone determina si el pasajero viaja completamente solo asignandole un 1 y con un 0 si va acompañado

,Ticket,Fare,GroupSize,Fare_Per_Person,FamilySize,IsAlone,Deck
0,A/5 21171,7.2500,1,7.25000,2,0,U
1,PC 17599,71.2833,1,71.28330,2,0,C
2,STON/O2. 3101282,7.9250,1,7.92500,1,1,U
3,113803,53.1000,2,26.55000,2,0,C
4,373450,8.0500,1,8.05000,1,1,U
5,330877,8.4583,1,8.45830,1,1,U
6,17463,51.8625,1,51.86250,1,1,E
7,349909,21.0750,4,5.26875,5,0,U
8,347742,11.1333,3,3.71110,3,0,U
9,237736,30.0708,2,15.03540,2,0,U


# Compensación de importes
Actualizaremos los importes de precios de billete en libras a un valor actual. En 1912, £1 libra esterlina equivalía a aproximadamente £130–£140 actuales debido a la inflación acumulada a lo largo de más de un siglo. Si convertimos la libra esterlina actual a Euros (€) (asumiendo un tipo de cambio estándar de ~$1.15–$1.20 por libra), £1 de 1912 equivale a unos €160 € de hoy.

In [14]:
# 1. Factor de conversión estimado: 1 GBP (1912) ≈ 160 EUR (Actuales)
FACTOR_INFLACION_EUR = 160.0

# 2. Calcular tarifa total ajustada a la inflación
df['Fare_EUR_Today'] = df['Fare'] * FACTOR_INFLACION_EUR

# 3. Calcular tarifa individual ajustada a la inflación
df['Fare_Per_Person_EUR_Today'] = df['Fare_Per_Person'] * FACTOR_INFLACION_EUR

# Mostrar estadísticas descriptivas por clase en Euros actuales
df.groupby('Pclass')['Fare_Per_Person_EUR_Today'].describe().round(2)

# Por que hay un valor de billete minimo de 0.0 ? El valor 0.0 representa pasajeros que realmente no pagaron
# el billete bien porque fueron empleados de la White Star Line o pasajeros de cortesia o con billetes de favor ( invitados y
# trabajadores basicamente)

,count,mean,std,min,25%,50%,75%,max
Pclass,,,,,,,,
1,216.0,6984.06,4851.56,0.0,4248.0,5640.00,8105.66,35484.67
2,184.0,2131.62,903.72,0.0,1680.0,2080.00,2217.39,5200.00
3,491.0,1293.74,383.03,0.0,1160.0,1256.67,1288.00,3604.00


# Altgoritmo y entrenaimiento del modelo
Llego el momento de entrenar el modelo y determinar que altgoritmo usar, me decidi por Random Forest basicamente por 3 razones:
- Nos da de forma nativa la importancia de las variables.
- Trabaja bien con realciones no lineales e iteraciones complejas, basicamente sabe que una mujer de 3 clase tiene una probabilildad de supervivencia muy diferente a una de 1 clase.
- Es bastante bueno trabajando con la cantidad de datos que estamos manejando, otros modelos podrian sobreentrenar haciendo que el modelo memorice mas que evalue (overfitting)

In [15]:
# 1. Seleccionar las columnas que usaremos para predecir
features = ['Pclass', 'Sex', 'Age', 'Fare_Per_Person', 'FamilySize', 'IsAlone', 'Title']

# 2. Convertir texto a números (One-Hot Encoding para Sex y Title)
X = pd.get_dummies(df[features], drop_first=True)
y = df['Survived']

# 3. Dividir datos: 80% para entrenar y 20% para evaluar la precisión
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Crear y entrenar el modelo Random Forest
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

#  con max_depth=5 creo que estamos en lo que suele llamarse el 'punto dulce' para la cantidad de datos que manejamos
#  con valores mayores obtendriamos un 100 % de precision con los datos de entrenamiento
#  pero lo que hace el modelo es memorizar asi que fallara estrepitosamente con datos reales
#  porque aprende en lugar de crear reglas generales

# 5. Evaluar la precisión del modelo con los datos de prueba. El 80% - 82% es un valor excelente para
#    titanic.csv
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Precisión del modelo en los datos de prueba: {accuracy * 100:.2f}%\n")

# 6. Guardar el modelo y los nombres de las columnas en archivos .pkl
joblib.dump(model, 'model_titanic.pkl')
joblib.dump(X.columns.tolist(), 'model_columns.pkl')

print("💾 Archivos 'model_titanic.pkl' y 'model_columns.pkl' guardados correctamente.")

✅ Precisión del modelo en los datos de prueba: 82.12%

💾 Archivos 'model_titanic.pkl' y 'model_columns.pkl' guardados correctamente.
